# 03 · Тревоги по типам датчиков по годам

**Цель** — понять, какие типы датчиков дают тревоги и как это меняется по годам. Это основа выбора направления прогнозирования и понимания стабильности целевого класса.

In [ ]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
if _HERE.name == "notebooks":
    RESEARCH = _HERE.parent
else:
    RESEARCH = _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import pandas as pd
import data_utils as du

RECOMPUTE = True
print("dataset:", du.find_dataset_dir())

# 3.0 · Агрегация тревог по типам датчиков (артефакт `alarms_by_type_year.csv`)

По каждому году: подсчёт тревог канала → merge со справочником типов → группировка по подсистеме и типу датчика.

In [ ]:
out_csv = du.find_dataset_dir() / "alarms_by_type_year.csv"
if RECOMPUTE or not out_csv.exists():
    ref_ch = du.load_ref_channels()[["ид_канала_данных", "тип_датчика", "тип_инж_системы"]]
    rows_all = []
    for fp in du.journal_files():
        year = fp.name.split("-")[2].split(".")[0]
        alarm_by_ch = {}
        for ch in du.read_chunks(fp, cols=["ид_канала_данных", "тревожное"]):
            al = ch.loc[ch["тревожное"] == "t", "ид_канала_данных"].value_counts()
            alarm_by_ch = pd.Series(alarm_by_ch).add(al, fill_value=0).astype("int64").to_dict()
        s = pd.Series(alarm_by_ch, dtype="int64").rename("alarms").to_frame() \
              .reset_index().rename(columns={"index": "ид_канала_данных"})
        s = s.merge(ref_ch, on="ид_канала_данных", how="left")
        s["тип_датчика"] = s["тип_датчика"].fillna("(нет в справочнике)")
        g = s.groupby(["тип_инж_системы", "тип_датчика"])["alarms"].sum()
        rows_all.append(pd.DataFrame({
            "год": year,
            "подсистема": g.index.get_level_values(0),
            "тип_датчика": g.index.get_level_values(1),
            "тревог": g.values,
        }))
        print(f"[{year}] ok", flush=True)
    at = pd.concat(rows_all)
    at.to_csv(out_csv, index=False, encoding="utf-8-sig")
else:
    at = pd.read_csv(out_csv, dtype={"год": str})
    print("Прочитано из готового артефакта:", out_csv)

print("\nТоп-20 (подсистема, тип) за все годы:")
print(at.groupby(["подсистема", "тип_датчика"])["тревог"].sum()
        .sort_values(ascending=False).head(20).to_string())

# 3.1 · Динамика ключевых типов по годам

Визуализация трендов для главных типов датчиков: дым, газ, тепло, температура, движение, двери, насосы, затопление.

In [ ]:
import matplotlib.pyplot as plt

key = ["Датчик дыма", "Газовый датчик", "Тепловой датчик", "Датчик температуры",
       "Датчик движения", "КД Дверь", "Состояние насоса", "Датчик затопления"]
wide = (at[at["тип_датчика"].isin(key)]
        .pivot_table(index="тип_датчика", columns="год",
                     values="тревог", aggfunc="sum", fill_value=0))
print(wide.to_string())

wide.T.plot(figsize=(12, 5), marker="o", title="Тревоги по типам датчиков по годам")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.0, 1.0))
plt.show()

# 3.2 · Комментарии

- **Датчик дыма — лидер тревог, но крайне нестабилен**: 38k из 63k пришлись на 2026 (янв–июнь). Либо массовые неисправности, либо смена методики.
- **Переключатель**: 355/375 тревог за всё время — в 2026. **Датчик температуры**: 10k/16k — в 2025. **Датчик движения**: пик 10 699 в 2025.
- Класс «тревога» **дрейфует** → модель без временной валидации переоценит качество. Обязателен временной сплит и, желательно, отдельные модели/калибровка по подсистемам.
- Для направления «отказ датчика» перспективнее не сам флаг `тревожное`, а **статусы неисправности** (см. ноутбук 04).